# 03 - Batch Inference

Loads the `champion` model from the Unity Catalog Model Registry and scores two groups of rows:
- **backtest**: test-year rows that already have an actual next quarter (so the error can be measured)
- **forward_forecast**: the latest quarter in the data, which forecasts the quarter after the data ends
  (only series that are still active in the latest quarter)

Predicted risk uses the **forecast**, not the historical `risk_label`:
1. predicted YoY growth <= -0.2 -> Declining
2. climate stress >= 30 -> Risk-prone
3. predicted YoY growth >= 0.2 -> High
4. otherwise -> Stable

Predicted YoY compares the forecast with the same quarter one year earlier (`production_lag_3`).
Output: `ml.crop_production_predictions`.

In [0]:
import mlflow
import numpy as np
from mlflow import MlflowClient
from pyspark.sql import functions as F

from configs.ml_config import (
    ALL_FEATURE_COLUMNS,
    BOOLEAN_FEATURES,
    CLIMATE_STRESS_THRESHOLD,
    DECLINE_THRESHOLD,
    FEATURE_TABLE,
    GROWTH_THRESHOLD,
    LABEL_COLUMN,
    MODEL_ALIAS,
    MODEL_NAME,
    PREDICTIONS_TABLE,
    NUMERIC_FEATURES,
    REGISTERED_MODEL_NAME,
    TEST_YEAR,
)
from common.delta_io import write_delta_table


output_schema = """
    province STRING, crop STRING, crop_group STRING,
    feature_year BIGINT, feature_quarter STRING,
    forecast_year BIGINT, forecast_quarter STRING, prediction_type STRING,
    actual_production DOUBLE, predicted_production DOUBLE, error DOUBLE,
    predicted_yoy DOUBLE, climate_stress DOUBLE, model_name STRING, model_version STRING
"""

In [0]:
mlflow.set_registry_uri("databricks-uc")

model_uri = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
model = mlflow.sklearn.load_model(model_uri)
model_version = MlflowClient().get_model_version_by_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS).version
print(f"Loaded {model_uri} (version {model_version})")

## 1. Select the rows to score

In [0]:
features_df = spark.table(FEATURE_TABLE)
latest_time_index = features_df.agg(F.max("time_index")).first()[0]

backtest_df = features_df.where(f"year = {TEST_YEAR} AND {LABEL_COLUMN} IS NOT NULL")
forward_df = features_df.where(f"time_index = {latest_time_index} AND {LABEL_COLUMN} IS NULL")

scoring_df = backtest_df.unionByName(forward_df).toPandas()
if scoring_df.empty:
    raise ValueError("Nothing to score.")

for column in NUMERIC_FEATURES + BOOLEAN_FEATURES:
    scoring_df[column] = scoring_df[column].astype("float")

## 2. Predict and derive the forecast columns (pandas)

In [0]:
scoring_df["predicted_production"] = np.asarray(model.predict(scoring_df[ALL_FEATURE_COLUMNS]), dtype=float)
scoring_df["actual_production"] = scoring_df[LABEL_COLUMN]
scoring_df["error"] = scoring_df["predicted_production"] - scoring_df["actual_production"]
scoring_df["prediction_type"] = np.where(scoring_df["actual_production"].notna(), "backtest", "forward_forecast")

# Climate stress = |rolling rain anomaly| + |rolling temperature anomaly|
scoring_df["climate_stress"] = scoring_df["rain_anomaly_rolling"].abs() + scoring_df["temp_anomaly_rolling"].abs()

# Predicted YoY: forecast vs. the same quarter one year earlier (3 quarters before the feature row)
last_year_value = scoring_df["production_lag_3"]
scoring_df["predicted_yoy"] = np.where(
    last_year_value.notna() & (last_year_value != 0),
    (scoring_df["predicted_production"] - last_year_value) / last_year_value,
    np.nan,
)

# The forecast is for the quarter AFTER the feature row. Q4 rolls over to Q1 of the next year.
is_q4 = scoring_df["quarter_num"] == 4
scoring_df["forecast_year"] = np.where(is_q4, scoring_df["year"] + 1, scoring_df["year"])
scoring_df["forecast_quarter"] = np.where(
    is_q4, "Q1", "Q" + (scoring_df["quarter_num"] + 1).astype(int).astype(str)
)
scoring_df["model_name"] = MODEL_NAME
scoring_df["model_version"] = str(model_version)

## 3. Save the predictions (risk label is added in PySpark)

In [0]:
output_df = scoring_df[[
    "province", "crop", "crop_group", "year", "quarter",
    "forecast_year", "forecast_quarter", "prediction_type",
    "actual_production", "predicted_production", "error",
    "predicted_yoy", "climate_stress", "model_name", "model_version",
]].rename(columns={"year": "feature_year", "quarter": "feature_quarter"})

predictions_df = spark.createDataFrame(output_df, output_schema).withColumn("scored_at", F.current_timestamp())

# pandas stores "missing" as NaN. Convert to a real NULL so IS NULL works in SQL.
for column in ["actual_production", "error", "predicted_yoy", "climate_stress"]:
    predictions_df = predictions_df.withColumn(column, F.nanvl(F.col(column), F.lit(None).cast("double")))

# Predicted risk (rules are listed at the top of this notebook). A missing YoY gives no label.
predictions_df = predictions_df.withColumn(
    "predicted_risk",
    F.when(F.col("predicted_yoy").isNull(), F.lit(None).cast("string"))
    .when(F.col("predicted_yoy") <= DECLINE_THRESHOLD, "Declining")
    .when(F.col("climate_stress") >= CLIMATE_STRESS_THRESHOLD, "Risk-prone")
    .when(F.col("predicted_yoy") >= GROWTH_THRESHOLD, "High")
    .otherwise("Stable"),
)

write_delta_table(predictions_df, PREDICTIONS_TABLE)

spark.table(PREDICTIONS_TABLE).groupBy("prediction_type", "predicted_risk").count().orderBy("prediction_type", "predicted_risk").show()